# 03 · Auditoria da base de Vidas

**Esteira PASA:** `01 Receita` → `02 Despesa` → `03 Vidas` → `04 Consolidado` → Power BI / `relatorio_executivo.html`

## Objetivo do ficheiro bruto

`data/Vidas.csv` traz a **quantidade de beneficiários** por segmento e competência. É o denominador do custo per capita e a base do perfil etário. Vidas representam **exposição mensal**, não pessoas únicas ao longo do ano.

Este caderno **não transforma nem grava** dados: valida tipos e integridade, descreve o perfil etário das carteiras e identifica a migração de vidas entre operadoras.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")


def localizar_raiz_projeto(inicio: Path | None = None) -> Path:
    """Sobe a arvore de diretorios ate encontrar a pasta ``data/`` do projeto."""
    inicio = (inicio or Path.cwd()).resolve()
    for pasta in (inicio, *inicio.parents):
        if (pasta / "data").is_dir():
            return pasta
    raise FileNotFoundError("Pasta 'data/' nao encontrada. Execute o caderno a partir da pasta do projeto.")


PASTA_RAIZ = localizar_raiz_projeto()
CHAVES_SEGMENTO = ["data_ref", "Operadora", "faixa_etaria", "uf", "cidade", "Plano"]

CAMINHO_VIDAS = PASTA_RAIZ / "data" / "Vidas.csv"
print(f"Fonte: {CAMINHO_VIDAS.relative_to(PASTA_RAIZ)}")

Fonte: data\Vidas.csv


In [2]:
def validar_schema(df: pd.DataFrame, esperado: dict[str, str]) -> pd.DataFrame:
    """Compara colunas e tipos lidos com o esperado; interrompe se houver divergencia."""
    relatorio = pd.DataFrame({
        "coluna": list(esperado),
        "tipo_esperado": list(esperado.values()),
        "tipo_lido": [str(df[c].dtype) if c in df else "AUSENTE" for c in esperado],
    })
    relatorio["confere"] = [
        c in df and (
            pd.api.types.is_string_dtype(df[c]) if tipo == "texto"
            else pd.api.types.is_integer_dtype(df[c]) if tipo == "inteiro"
            else pd.api.types.is_float_dtype(df[c]) if tipo == "decimal"
            else False
        )
        for c, tipo in esperado.items()
    ]
    assert list(df.columns) == list(esperado), f"Colunas divergentes: {list(df.columns)}"
    assert relatorio["confere"].all(), "Ha colunas com tipo diferente do esperado."
    return relatorio


def diagnosticar_integridade(df: pd.DataFrame, chave: list[str] | None) -> pd.Series:
    """Nulos, duplicidades, cobertura temporal e unicidade da chave de segmento."""
    datas = pd.to_datetime(df["data_ref"], format="%Y-%m-%d", errors="coerce")
    return pd.Series({
        "linhas": len(df),
        "celulas_nulas": int(df.isna().sum().sum()),
        "linhas_duplicadas_integralmente": int(df.duplicated().sum()),
        "segmentos_repetidos": int(df.duplicated(subset=chave).sum()) if chave else "n/a",
        "datas_invalidas": int(datas.isna().sum()),
        "primeira_competencia": datas.min().strftime("%m/%Y"),
        "ultima_competencia": datas.max().strftime("%m/%Y"),
        "competencias": datas.nunique(),
        "operadoras": ", ".join(sorted(df["Operadora"].unique())),
    }, name="valor")

## 1. Carga e validação de tipos

Dimensões como texto e `Vidas` como inteiro.

In [3]:
df_vidas = pd.read_csv(CAMINHO_VIDAS)

SCHEMA_VIDAS = {
    "Indicador": "texto", "data_ref": "texto", "faixa_etaria": "texto", "uf": "texto",
    "cidade": "texto", "Operadora": "texto", "Plano": "texto", "Vidas": "inteiro",
}
display(validar_schema(df_vidas, SCHEMA_VIDAS))
display(df_vidas.head())

,coluna,tipo_esperado,tipo_lido,confere
0,Indicador,texto,str,True
1,data_ref,texto,str,True
2,faixa_etaria,texto,str,True
3,uf,texto,str,True
4,cidade,texto,str,True
5,Operadora,texto,str,True
6,Plano,texto,str,True
7,Vidas,inteiro,int64,True


,Indicador,data_ref,faixa_etaria,uf,cidade,Operadora,Plano,Vidas
0,Vidas,2026-03-01,29 - 33,PA,PARAUAPEBAS,Operadora X,5F9A42D451,700
1,Vidas,2025-12-01,59 e acima,SP,SAO PAULO,Operadora Y,DBBECEB087,62
2,Vidas,2025-11-01,24 - 28,PA,PARAUAPEBAS,Operadora X,107354F3E8,982
3,Vidas,2026-02-01,0 - 18,MA,SAO LUIS,Operadora X,BF1F47E31D,472
4,Vidas,2025-09-01,29 - 33,ES,SERRA,Operadora X,107354F3E8,264


## 2. Integridade estrutural

In [4]:
integridade = diagnosticar_integridade(df_vidas, CHAVES_SEGMENTO)
display(integridade.to_frame())

assert integridade["celulas_nulas"] == 0, "Ha valores nulos."
assert integridade["linhas_duplicadas_integralmente"] == 0, "Ha linhas duplicadas."
assert integridade["segmentos_repetidos"] == 0, "Ha segmentos repetidos."
assert integridade["datas_invalidas"] == 0, "Ha datas invalidas."
assert integridade["competencias"] == 12, "Esperadas 12 competencias (08/2025 a 07/2026)."
assert df_vidas["Indicador"].eq("Vidas").all()
assert df_vidas["Vidas"].gt(0).all(), "Ha segmentos com vidas zeradas ou negativas."

,valor
linhas,215764
celulas_nulas,0
linhas_duplicadas_integralmente,0
segmentos_repetidos,0
datas_invalidas,0
primeira_competencia,08/2025
ultima_competencia,07/2026
competencias,12
operadoras,"Operadora X, Operadora Y"


## 3. Evolução mensal da carteira

In [5]:
vidas_mes = df_vidas.pivot_table(index="data_ref", columns="Operadora", values="Vidas", aggfunc="sum")
vidas_mes_delta = vidas_mes.diff().add_suffix(" (variacao)")
display(pd.concat([vidas_mes, vidas_mes_delta], axis=1))

variacao_pct = (vidas_mes.iloc[-1] / vidas_mes.iloc[0] - 1) * 100
porte = vidas_mes.mean()
print("Variacao ponta a ponta (%):", variacao_pct.round(2).to_dict())
print(f"Porte medio X/Y: {porte['Operadora X'] / porte['Operadora Y']:.1f}x")

Operadora,Operadora X,Operadora Y,Operadora X (variacao),Operadora Y (variacao)
data_ref,,,,
2025-08-01,402180,77578,NaN,NaN
2025-09-01,401670,77430,-510.00,-148.00
2025-10-01,402308,77102,638.00,-328.00
2025-11-01,403324,76710,"1,016.00",-392.00
2025-12-01,404184,76336,860.00,-374.00
2026-01-01,393846,86382,"-10,338.00","10,046.00"
2026-02-01,393580,86078,-266.00,-304.00
2026-03-01,395056,86174,"1,476.00",96.00
2026-04-01,393692,86478,"-1,364.00",304.00


Variacao ponta a ponta (%): {'Operadora X': -2.05, 'Operadora Y': 12.42}
Porte medio X/Y: 4.8x


## 4. Achado crítico (a) — perfil etário das carteiras

Participação de cada faixa etária na exposição total (vidas-mês) de cada operadora.

In [6]:
perfil_etario = df_vidas.pivot_table(index="faixa_etaria", columns="Operadora", values="Vidas", aggfunc="sum")
perfil_etario = perfil_etario / perfil_etario.sum() * 100
display(perfil_etario)

assert round(perfil_etario.at["59 e acima", "Operadora Y"], 2) == 48.51
assert round(perfil_etario.at["0 - 18", "Operadora X"], 2) == 32.52

Operadora,Operadora X,Operadora Y
faixa_etaria,,
0 - 18,32.52,11.24
19 - 23,7.12,3.45
24 - 28,4.42,5.14
29 - 33,7.43,4.93
34 - 38,11.37,5.13
39 - 43,13.40,5.40
44 - 48,10.73,6.47
49 - 53,5.52,5.03
54 - 58,3.00,4.71


**Evidência:** a **Operadora Y** é uma carteira envelhecida — **48,51% da exposição está na faixa 59+**. A **Operadora X** é jovem — 32,52% na faixa 0–18 e apenas 4,49% em 59+.

**Risco:** comparar o custo per capita bruto entre as duas penaliza a carteira mais velha e esconde ineficiências da mais jovem.

**Recomendação / tratamento no caderno 04:** padronização direta por idade (seção 6.4). Com a pirâmide etária da X, a Y custaria R$ 111,15 por vida/mês, abaixo da X (R$ 124,39).

## 5. Achado crítico (b) — migração de vidas em jan/2026

A tabela da seção 3 mostra saltos de sinal oposto e magnitude semelhante (cerca de 10 mil vidas) na mesma competência. Investigamos quais planos explicam o movimento.

In [7]:
LIMIAR_MOVIMENTO_PLANO = 1_000  # vidas

vidas_plano = df_vidas.pivot_table(index=["Operadora", "Plano"], columns="data_ref", values="Vidas",
                                   aggfunc="sum", fill_value=0)
saltos = vidas_mes.diff().abs().idxmax()
MES_QUEBRA = saltos.mode().iat[0]
MES_ANTERIOR = vidas_plano.columns[list(vidas_plano.columns).index(MES_QUEBRA) - 1]
print(f"Competencia da quebra: {MES_QUEBRA} (comparada a {MES_ANTERIOR})")

movimento = pd.DataFrame({"vidas_antes": vidas_plano[MES_ANTERIOR], "vidas_depois": vidas_plano[MES_QUEBRA]})
movimento["variacao"] = movimento["vidas_depois"] - movimento["vidas_antes"]
movimento = movimento[movimento["variacao"].abs() >= LIMIAR_MOVIMENTO_PLANO]
display(movimento)

assert MES_QUEBRA == "2026-01-01"
assert set(movimento.index) == {("Operadora X", "340154BC84"), ("Operadora Y", "0F2726E6AC")}

Competencia da quebra: 2026-01-01 (comparada a 2025-12-01)


,,vidas_antes,vidas_depois,variacao
Operadora,Plano,,,
Operadora X,340154BC84,10110,8,-10102
Operadora Y,0F2726E6AC,0,10244,10244


In [8]:
def perfil_plano(plano: str, competencia: str) -> pd.Series:
    filtro = df_vidas["Plano"].eq(plano) & df_vidas["data_ref"].eq(competencia)
    distribuicao = df_vidas[filtro].groupby("faixa_etaria")["Vidas"].sum()
    return distribuicao / distribuicao.sum() * 100


comparacao = pd.DataFrame({
    "X 340154BC84 (12/2025) %": perfil_plano("340154BC84", MES_ANTERIOR),
    "Y 0F2726E6AC (01/2026) %": perfil_plano("0F2726E6AC", MES_QUEBRA),
}).fillna(0)
similaridade = 100 - comparacao.diff(axis=1).iloc[:, -1].abs().sum() / 2
display(comparacao)

uf_planos = df_vidas[df_vidas["Plano"].isin(["340154BC84", "0F2726E6AC"])].groupby("Plano")["uf"].unique()
print(f"Similaridade da distribuicao etaria: {similaridade:.1f}%")
print("UF dos planos:", uf_planos.to_dict())
assert round(similaridade, 1) == 99.0

,X 340154BC84 (12/2025) %,Y 0F2726E6AC (01/2026) %
faixa_etaria,,
0 - 18,33.97,33.72
19 - 23,7.81,8.82
24 - 28,4.04,4.06
29 - 33,6.79,6.79
34 - 38,9.77,9.61
39 - 43,10.94,10.89
44 - 48,11.26,11.05
49 - 53,6.94,6.79
54 - 58,4.85,4.71


Similaridade da distribuicao etaria: 99.0%
UF dos planos: {'0F2726E6AC': <StringArray>
['PA', 'MG', 'DF', 'BA', 'PR', 'MA', 'RJ', 'ES', 'SP', 'CE', 'AM', 'PE', 'RN',
 'PB', 'MT']
Length: 15, dtype: str, '340154BC84': <StringArray>
['PA', 'MG', 'PR', 'DF', 'RJ', 'ES', 'SP', 'GO', 'BA', 'MA', 'PE', 'AM', 'MT',
 'PB', 'RN', 'CE']
Length: 16, dtype: str}


**Evidência:** em jan/2026 o plano `340154BC84` da X (PA) cai de 10.110 para 8 vidas e, na mesma competência, nasce o plano `0F2726E6AC` da Y (PA) com 10.244 vidas e distribuição etária **99,0% semelhante**. Mesma data, mesmo porte, mesma UF, mesmo perfil: o padrão é de **transferência de carteira**, não de movimento comercial.

**Risco:** ler o +12,42% de vidas da Y e o −2,05% da X como tendência comercial. Sem o bloco transferido, a X cresce 0,52% e a Y encolhe 0,84%. O plano chega à Y **sem receita na base** (ver caderno 01).

**Recomendação / tratamento no caderno 04:** separar crescimento orgânico e inorgânico (seção 5) e confirmar contratualmente a operação (portabilidade, cessão de carteira ou migração de contrato) e a responsabilidade pela cauda de sinistros.

## 6. Conclusões da auditoria

| Verificação | Resultado |
|---|---|
| Colunas e tipos | Conforme o esperado (7 dimensões texto + `Vidas` inteiro) |
| Nulos, duplicados, datas inválidas | Nenhum |
| Chave de segmento | Única — sem segmentos repetidos |
| Cobertura | Operadoras X e Y, 12 competências (08/2025 a 07/2026); X ≈ 4,8× a Y |
| Perfil etário | Y envelhecida (59+ = 48,51%); X jovem (0–18 = 32,52%) |
| Migração | ~10 mil vidas da X → Y em jan/2026 (planos `340154BC84` → `0F2726E6AC`) |